In [0]:
 import pandas as pd
import numpy as np
import datetime

def generate_fintech_data(num_users=1000, months=12):
    np.random.seed(42)
    start_date = datetime.date(2025, 1, 1)
    data = []
    
    # 1. Generate core user profiles
    for user_id in range(1000, 1000 + num_users):
        # Evenly spread cohort start months across the year
        cohort_month = np.random.randint(0, months)
        join_date = start_date + datetime.timedelta(days=cohort_month * 30)
        
        # Assign base tier subscription value
        base_arr = np.random.choice([299, 599, 999]) 
        churn_probability = np.random.uniform(0.02, 0.08) # 2% to 8% baseline monthly churn
        
        # 2. Simulate user life lifecycle behavior across 12 months
        is_churned = False
        for m in range(cohort_month, months):
            if is_churned:
                continue
                
            current_month_date = start_date + datetime.timedelta(days=m * 30)
            
            # Simulate natural churn occurrence
            if np.random.random() < (churn_probability * (m - cohort_month + 1) * 0.5):
                is_churned = True
                continue
            
            # Simulate strategic fintech expansions (Upgrades/Downgrades)
            event_type = "Standard Renewal"
            monthly_amount = base_arr
            rand_val = np.random.random()
            
            if rand_val > 0.92: # 8% chance of tier expansion
                monthly_amount = base_arr * 1.3
                event_type = "Tier Upgrade"
            elif rand_val < 0.04: # 4% risk of downgrades
                monthly_amount = base_arr * 0.7
                event_type = "Contraction"
                
            data.append({
                "transaction_id": f"TXN-{user_id:04d}-{m:02d}",
                "user_id": f"USER-{user_id}",
                "cohort_group": join_date.strftime("%Y-%m"),
                "transaction_date": current_month_date.strftime("%Y-%m-%d"),
                "mrr_amount": round(monthly_amount, 2),
                "event_type": event_type
            })
            
    df = pd.DataFrame(data)
    df.to_csv("mock_fintech_transactions.csv", index=False)
    print(f"🎉 Success! Generated {len(df)} transactional ledger events across {num_users} unique user accounts.")
    print("Saved directly to: 'mock_fintech_transactions.csv'")

if __name__ == "__main__":
    generate_fintech_data()


In [0]:
import pandas as pd
import numpy as np

def calculate_nrr_matrix(csv_path="mock_fintech_transactions.csv"):
    # 1. Load the ledger data
    df = pd.read_csv(csv_path)
    
    # 2. Parse dates and calculate the "Cohort Index" (months elapsed since sign-up)
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    df['cohort_group'] = pd.to_datetime(df['cohort_group'] + "-01")
    
    # Calculate months between transaction date and initial cohort sign-up month
    df['cohort_index'] = ((df['transaction_date'].dt.year - df['cohort_group'].dt.year) * 12 + 
                          (df['transaction_date'].dt.month - df['cohort_group'].dt.month))
    
    # 3. Pivot data to aggregate Monthly Recurring Revenue (MRR) per cohort per month
    cohort_pivot = df.pivot_table(
        index='cohort_group',
        columns='cohort_index',
        values='mrr_amount',
        aggfunc='sum'
    )
    
    # 4. Calculate Net Revenue Retention (NRR) Matrix
    # Month 0 (the index column 0) represents the starting base baseline revenue (100%)
    cohort_sizes = cohort_pivot.iloc[:, 0]
    nrr_matrix = cohort_pivot.divide(cohort_sizes, axis=0) * 100
    
    # Clean up formatting for executive presentations
    nrr_matrix.index = nrr_matrix.index.strftime('%Y-%m')
    
    # 5. Export summary files
    nrr_matrix.round(1).to_csv("fintech_nrr_matrix.csv")
    print("🚀 Success! Raw ledger records converted into an Executive NRR Percentage Matrix.")
    print("Saved directly to: 'Data/fintech_nrr_matrix.csv'\n")
    
    # Display snapshot of the matrix
    print("--- 📊 NRR MATRIX SNAPSHOT (First 5 Cohorts, First 6 Months) ---")
    print(nrr_matrix.round(1).iloc[:5, :6].to_string())

if __name__ == "__main__":
    # If running locally, adjust path if your data is sitting inside the /Data folder
    try:
        calculate_nrr_matrix("mock_fintech_transactions.csv")
    except FileNotFoundError:
        calculate_nrr_matrix("Data/mock_fintech_transactions.csv")


In [0]:
# Cell 3: Add your analysis code here

import pandas as pd
import numpy as np

# 1. Load the ledger data you generated earlier
# Note: If your file is inside a 'Data' folder, change this to "Data/mock_fintech_transactions.csv"
try:
    df = pd.read_csv("mock_fintech_transactions.csv")
except FileNotFoundError:
    df = pd.read_csv("Data/mock_fintech_transactions.csv")

# 2. Parse dates to ensure calculations work properly
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df['cohort_group'] = pd.to_datetime(df['cohort_group'] + "-01")

# Calculate the "Cohort Index" (how many months have passed since the user signed up)
df['cohort_index'] = ((df['transaction_date'].dt.year - df['cohort_group'].dt.year) * 12 + 
                      (df['transaction_date'].dt.month - df['cohort_group'].dt.month))

# 3. Pivot data to calculate total Monthly Recurring Revenue (MRR) per cohort per month
cohort_pivot = df.pivot_table(
    index='cohort_group',
    columns='cohort_index',
    values='mrr_amount',
    aggfunc='sum'
)

# 4. Calculate the Net Revenue Retention (NRR) Matrix as a percentage
# Month 0 acts as our 100% baseline value for each cohort group
cohort_sizes = cohort_pivot.iloc[:, 0]
nrr_matrix = cohort_pivot.divide(cohort_sizes, axis=0) * 100

# Clean up the row headers to look clean (YYYY-MM)
nrr_matrix.index = nrr_matrix.index.strftime('%Y-%m')

# 5. Output a clean executive snapshot directly into your notebook view
print("🎉 DATA PIPELINE SUCCESSFUL!")
print("Below is your processed Net Revenue Retention (NRR) Matrix.\n")
print("--- 📊 STRATEGIC NRR MATRIX SNAPSHOT (First 6 Months) ---")
print(nrr_matrix.round(1).iloc[:, :6].to_string())
